# 04 — Job Clustering & Skill-Gap Report (Unsupervised)

Discover natural job families using **K-Means** and generate personalised **skill-gap reports**.

In [ ]:
import sys
sys.path.insert(0,"..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
import joblib
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Load Job Matrix & Vectorizer

In [ ]:
from src.config import TFIDF_PATH, MODELS_DIR, JOB_CORPUS_CSV
from src.features.text_features import load_vectorizer

vectorizer = load_vectorizer()
job_matrix = load_npz(str(MODELS_DIR/"job_tfidf_matrix.npz"))
job_corpus  = pd.read_csv(JOB_CORPUS_CSV)
print("Job matrix:", job_matrix.shape)

## 2. Elbow Method + Silhouette Analysis

Use the elbow method and silhouette scores to pick the optimal **k**.

In [ ]:
from src.models.clustering import elbow_analysis
results = elbow_analysis(job_matrix, k_range=range(2,15), save_fig=True)
print(pd.DataFrame({"k":results["k"],"inertia":results["inertia"],"silhouette":results["silhouette"]}))

## 3. Train K-Means with Optimal k

In [ ]:
from src.models.clustering import train_kmeans
from src.config import N_CLUSTERS

# Adjust N_CLUSTERS in src/config.py based on elbow analysis
km, labels = train_kmeans(job_matrix, n_clusters=N_CLUSTERS)

job_corpus["cluster"] = labels
print("
Cluster sizes:")
print(job_corpus["cluster"].value_counts().sort_index())

## 4. Visualise Clusters (PCA)

In [ ]:
from src.models.clustering import plot_clusters
plot_clusters(job_matrix, labels, method="pca", save=True)

## 5. Top Skills Per Cluster

In [ ]:
from src.models.clustering import cluster_top_skills
c_skills = cluster_top_skills(job_matrix, labels, vectorizer, n_top=15)

for cid, skills in sorted(c_skills.items()):
    print(f"
Cluster {cid}: {skills[:10]}")

## 6. Sample Resumes in Each Cluster

In [ ]:
for cid in sorted(job_corpus["cluster"].unique())[:5]:
    sample_jobs = job_corpus[job_corpus["cluster"]==cid]["title"].head(5).tolist()
    print(f"
Cluster {cid}: {sample_jobs}")

## 7. Skill-Gap Report for a Sample Resume

In [ ]:
from src.data.preprocess import clean_text
from src.models.clustering import skill_gap_report, assign_cluster
from src.features.text_features import transform_tfidf

sample_resume = """
Data Analyst with 3 years of experience. Proficient in SQL, Excel, Tableau,
and Python (pandas, numpy). Experience with data cleaning and reporting.
Want to move into Data Science roles.
"""

clean = clean_text(sample_resume)
resume_vec = transform_tfidf(vectorizer, [clean])
target_cluster = assign_cluster(resume_vec, km)
gap = skill_gap_report(clean, target_cluster, c_skills)

print(f"
Target Cluster:          {gap['target_cluster']}")
print(f"Cluster Top Skills:      {gap['cluster_top_skills'][:10]}")
print(f"Skills You Have:         {gap['resume_skills_found']}")
print(f"Skills to Develop:       {gap['missing_skills'][:10]}")
print(f"Skill Match:             {gap['match_pct']}%")

## 8. Skill-Gap Bar Chart

In [ ]:
import matplotlib.patches as mpatches

skills     = gap["cluster_top_skills"]
has_skill  = [1 if s in gap["resume_skills_found"] else 0 for s in skills]
colors     = ["#34d399" if h else "#ef4444" for h in has_skill]

fig, ax = plt.subplots(figsize=(10,6))
ax.barh(skills[::-1], [1]*len(skills), color=colors[::-1], alpha=0.8)
ax.set_xlabel("Skill Present")
ax.set_title(f"Skill-Gap Report — Cluster {target_cluster}")
green  = mpatches.Patch(color="#34d399", label="Have")
red    = mpatches.Patch(color="#ef4444", label="Missing")
ax.legend(handles=[green, red])
plt.tight_layout()
plt.savefig("../reports/figures/skill_gap.png", dpi=150)
plt.show()

## ✅ Clustering complete. Next → 05_fit_predictor.ipynb (optional)